In [4]:
import glob
import json
import os

import numpy as np
import pandas as pd

In [5]:
datasets = [
    'hmof', 'hmofheat',
    'cof', 'ppn', 'zeolite',
    'c3h6c3h8coremof', 'ch4n2',
    'corecof',
]

In [6]:
ds_tasks = {
    ds: [i.split('-to_')[-1] for i in glob.glob(f'../experiments/evaluate/few_shot/{ds}/*scratch*')]
    for ds in datasets
}

In [13]:
pretty_labels = {
    # Models
    'cgcnn': 'CGCNN',
    'moformer': 'MOFormer',
    'pmtransformer': 'PMTransformer',
    'spbnet': 'SpbNet',
    'intellipore': 'IntelliPore',
    # Datasets,
    'hmof': 'hMOF',
    'hmofheat': 'hMOF',
    'cof': 'hCOF',
    'zeolite': 'hZEO',
    'ppn': 'hPPN',
    'ch4n2': 'CoRE 2019',
    'c3h6c3h8coremof': 'CoRE 2019',
    'corecof': 'CoRE COF',
    'uo_water': 'UO',
    # hmofheat
    'H2-77K-2bar': 'H$_2$ heat of adsorption at 77 K and 2 bar (MOFs)',
    'CO2-298K-2.5bar': 'CO$_2$ heat of adsorption at 298 K and 2.5 bar (MOFs)',
    # hmof
    'H2-77-2': 'H$_2$ uptake at 77 K and 2 bar (MOFs)',
    'CO2-298-2.5': 'CO$_2$ uptake at 298 K and 2.5 bar (MOFs)',
    'CH4-298-2.5': 'CH$_4$ uptake at 298 K and 2.5 bar (MOFs)',
    'N2-298-0.9': 'N$_2$ uptake at 298 K and 0.9 bar (MOFs)',
    'Xe-273-10': 'Xe binary uptake at 273 K and 10 bar (MOFs)',
    'Kr-273-10': 'Kr binary uptake at 273 K and 10 bar (MOFs)',
    # cof
    'lowbar': 'CH$_4$ uptake at 298 K and 5.8 bar (COFs)',
    'highbar': 'CH$_4$ uptake at 298 K and 65 bar (COFs)',
    # ppn
    '1bar': 'CH$_4$ uptake at 298 K and 1 bar (PPNs)',
    '65bar': 'CH$_4$ uptake at 298 K and 65 bar (PPNs)',
    # zeolite
    'unitless_KH': ' CH$_4$ (unitless) Henry coefficient at 298 K (ZEOs)',
    'heat_of_adsorption': 'CH$_4$ isosteric heat of adsorption at 298 K (ZEOs)',
    # c3h6c3h8coremof
    'C3H6_loadings': 'C$_3$H$_6$ binary uptake at 298 K and 1 bar (MOFs)',
    'C3H8_loadings': 'C$_3$H$_8$ binary uptake at 298 K and 1 bar (MOFs)',
    'C3H8_C3H6_Selectivity_1bar': 'C$_3$H$_8$/C$_3$H$_6$ selectivity at 298 K and 1 bar (MOFs)',
    'C3H8_C3H6_Selectivity_infinite': 'C$_3$H$_8$/C$_3$H$_6$ infinite dillution selectivity at 298 K (MOFs)',
    'C3H6_Henry_298K': 'C$_3$H$_6$ (log) Henry coefficient at 298 K (MOFs)',
    'C3H8_Henry_298K': 'C$_3$H$_8$ (log) Henry coefficient at 298 K (MOFs)',
    'TSN_S_1Bar': r'$N_\mathrm{C_3H_8} \times \log(S_\mathrm{C_3H_8/C_3H_6})$ at 298 K and 1 bar (MOFs)',
    # ch4n2
    'ch4n2ratio-0.1bar': 'CH$_4$/N$_2$ selectivity and 298 K and 0.1 bar (MOFs)',
    'ch4n2ratio-10bar': 'CH$_4$/N$_2$ selectivity and 298 K and 10 bar (MOFs)',
    'ch4n2ratio-1bar': 'CH$_4$/N$_2$ selectivity and 298 K and 1 bar (MOFs)',
    'ch4uptake-0.1bar': 'CH$_4$ binary uptake at 298 K and 0.1 bar (MOFs)',
    'ch4uptake-10bar': 'CH$_4$ binary uptake at 298 K and 10 bar (MOFs)',
    'ch4uptake-1bar': 'CH$_4$ binary uptake at 298 K and 1 bar (MOFs)',
    'n2uptake-0.1bar': 'N$_2$ binary uptake at 298 K and 0.1 bar (MOFs)',
    'n2uptake-10bar': 'N$_2$ binary uptake at 298 K and 10 bar (MOFs)',
    'n2uptake-1bar': 'N$_2$ binary uptake at 298 K and 1 bar (MOFs)',
    # corecof
    'H2_log_Henry': 'H$_2$ log Henry coefficient at 298 K (COFs)',
    'O2_log_Henry': 'O$_2$ log Henry coefficient at 298 K (COFs)',
    'N2_log_Henry': 'N$_2$ log Henry coefficient at 298 K (COFs)',
    'CH4_log_Henry': 'CH$_4$ log Henry coefficient at 298 K (COFs)',
    'CO2_log_Henry': 'CO$_2$ log Henry coefficient at 298 K (COFs)',
    # uo_water
    'log_henry_coefficient_H2O_298K [mol_kg_bar]': 'H$_2$O log Henry coefficient at 298 K (MOFs)',
}

In [14]:
tex_template = r'''
\begin{{figure}}[!h]
    \centering
    \includegraphics[width=0.6\textwidth]{{fig/few_shot/{ds_task}_test_R2Score.pdf}}
    \includegraphics[width=0.6\textwidth]{{fig/few_shot/{ds_task}_test_MeanAbsoluteError.pdf}}
    \caption{{Comparison of sample efficiency between training from scratch and
    fine-tuning. Results are shown as mean $\pm$ standard deviation of different
    runs.}}
\end{{figure}}

\begin{{table}}[!h]
\centering
{table_r2}{table_mae}
\caption{{Average $R^2$ (top) and MAE (bottom) of different runs is reported.}}
\end{{table}}
'''

In [15]:
def load_json(path):
    with open(path) as fhand:
        return json.load(fhand)

In [16]:
def to_latex(dataset):
    
    ckpt_paths = [
        'from_scratch-to_',
        'from_experiments_pretrain_early_pretrain_lightning_logs_version_0_checkpoints_best.ckpt-to_',
    ]

    if dataset != 'corecof':
        sizes_runs = {'50': '10', '100': '10', '300': '5', '500': '5', '1000': '3'}
    else:
        sizes_runs = {'50': '10', '100': '10', '300': '5', '500': '5', '800': '3'}

    for task in ds_tasks[dataset]:

        tex_tables = {}
        
        for metric in ['test_R2Score', 'test_MeanAbsoluteError']:
            table = {}
            ds_task = f'{dataset}_{task}'
    
            for ckpt, label in zip(ckpt_paths, ['Scratch training', 'Fine-tuning']):
                json_path = f'../experiments/evaluate/few_shot/{dataset}/{ckpt}{task}/results.json'

                results = load_json(json_path)

                table[label] = [
                    np.mean([results[size][str(run)][metric] for run in range(int(n_runs))])
                    for size, n_runs in sizes_runs.items()
                ]

            header = True if metric == 'test_R2Score' else False
            
            table = pd.DataFrame(table, index=[rf'\textbf{{{s}}}' for s in sizes_runs.keys()]).T
            table = table.to_latex(float_format="%.3f", header=header)

            if metric == 'test_R2Score':
                table = table.replace('begin{tabular}{lrrrrr}', 'begin{tabularx}{\\textwidth}{Xccccc}')
                table = table.replace('\\end{tabular}\n', '')
                table = table.replace('\\bottomrule\n', '')
                table = table.replace(r' & \textbf{50}', rf'\textbf{{{pretty_labels[task]}}} & \textbf{{50}}')
            else:
                table = table.replace('\\begin{tabular}{lrrrrr}\n', '')
                table = table.replace('\\midrule\n', '')
                table = table.replace('toprule', 'midrule')
                table = table.replace('end{tabular}', 'end{tabularx}')
            
            tex_tables[metric] = table

        output = tex_template.format(
            dataset=f'{pretty_labels[dataset]} dataset',
            ds_task=ds_task,
            table_r2=tex_tables['test_R2Score'],
            table_mae=tex_tables['test_MeanAbsoluteError'],
        )
        print(output)

In [17]:
%%capture cap
for ds in datasets:
    # Group identifiers under same dataset
    if ds not in ['hmofheat', 'ch4n2']:
        print(f'\\subsection{{{pretty_labels[ds]} dataset}}')
    to_latex(ds)
    print('\\clearpage')

In [18]:
with open('few_shot.tex', 'w') as out:
    out.write(cap.stdout)